# NEO-NN → ONNX conversion

Exports each member of the NEO-NN BSON ensembles to ONNX (mirrors
`TurbulentTransport/utilities/convert_nn.ipynb`). Each model gets a
subdirectory `models/<name>/` with 20 `*_model_<i>.onnx` files plus sidecar
text files (`xnames`, `ynames`, `xm`, `xsigma`, `ym`, `ysigma`,
`xbounds_min`, `xbounds_max`).

The exported ONNX graphs are the **raw networks**: consumers must apply the
input pipeline themselves —
`log10` on any `*_log10` feature (only `NU_1_log10`), then `(x - xm)/xsigma` —
and de-standardize the output with `y*ysigma + ym`. ONNX uses batch-first
layout `[N_samples, N_features]` (transposed w.r.t. the Julia side).

In [ ]:
using Pkg
Pkg.activate(; temp=true)
Pkg.develop(; path=dirname(pwd()))   # run from utilities/
Pkg.add(["ONNXNaiveNASflux", "Flux", "ONNXRunTime"])
using NeoclassicalTransport, Flux, ONNXNaiveNASflux
const NCT = NeoclassicalTransport

In [ ]:
# Export all 6 ensembles (drop entries from `names` to convert a subset)
names = ["neonn_tgyro_$(dev)_$(grp)_v1" for dev in ("d3d", "mastu+nstx", "d3d+mastu+nstx") for grp in ("flux", "flow")]

writevec(path, v) = open(path, "w") do io
    for x in v
        println(io, x)
    end
end

for name in names
    ens = NCT.loadmodelonce(name)
    outdir = mkpath(joinpath(dirname(pwd()), "models", name))
    for (i, model) in enumerate(ens.models)
        ONNXNaiveNASflux.save(joinpath(outdir, "$(name)_model_$i.onnx"), Flux.f32(model.fluxmodel))
    end
    writevec(joinpath(outdir, "xnames.txt"), ens.xnames)
    writevec(joinpath(outdir, "ynames.txt"), ens.ynames)
    writevec(joinpath(outdir, "xm.txt"), Float32.(ens.xm))
    writevec(joinpath(outdir, "xsigma.txt"), Float32.(ens.xσ))
    writevec(joinpath(outdir, "ym.txt"), Float32.(ens.ym))
    writevec(joinpath(outdir, "ysigma.txt"), Float32.(ens.yσ))
    writevec(joinpath(outdir, "xbounds_min.txt"), Float32.(ens.xbounds[:, 1]))
    writevec(joinpath(outdir, "xbounds_max.txt"), Float32.(ens.xbounds[:, 2]))
    println("$name: exported $(length(ens.models)) members + sidecars")
end

In [ ]:
# Verify: ONNXRunTime ensemble mean vs the Julia ensemble, at the center of
# the training space. Expect Float32-level agreement (~1e-5 relative).
using ONNXRunTime

for name in names
    ens = NCT.loadmodelonce(name)
    outdir = joinpath(dirname(pwd()), "models", name)
    nx = length(ens.xnames)

    xt = Float32.(ens.xm)              # xm is in log10 space for NU_1_log10
    xlin = copy(Float64.(xt))          # Julia side takes LINEAR values
    for (i, n) in enumerate(ens.xnames)
        endswith(n, "_log10") && (xlin[i] = 10.0^xlin[i])
    end
    y_julia = NCT.flux_array(ens, xlin; warn_nn_train_bounds=false)

    xn = (xt .- Float32.(ens.xm)) ./ Float32.(ens.xσ)   # ONNX nets are raw: pre-normalize
    acc = zeros(Float32, length(ens.ynames))
    for i in 1:length(ens.models)
        sess = ONNXRunTime.load_inference(joinpath(outdir, "$(name)_model_$i.onnx"))
        out = sess(Dict(only(ONNXRunTime.input_names(sess)) => permutedims(reshape(xn, nx, 1))))
        acc .+= vec(permutedims(out[only(ONNXRunTime.output_names(sess))]))
    end
    y_onnx = (acc ./ length(ens.models)) .* Float32.(ens.yσ) .+ Float32.(ens.ym)
    reldiff = maximum(abs.(y_onnx .- y_julia) ./ (abs.(y_julia) .+ 1e-10))
    println(rpad(name, 42), "max rel diff = ", round(reldiff; sigdigits=3))
end

## Python usage sketch

```python
import numpy as np, onnxruntime as ort

d = "models/neonn_tgyro_d3d+mastu+nstx_flux_v1"
xnames = open(f"{d}/xnames.txt").read().split()
xm, xs = (np.loadtxt(f"{d}/{f}.txt", dtype=np.float32) for f in ("xm", "xsigma"))
ym, ys = (np.loadtxt(f"{d}/{f}.txt", dtype=np.float32) for f in ("ym", "ysigma"))

x = ...                                # raw linear features, ordered like xnames
for i, n in enumerate(xnames):
    if n.endswith("_log10"):
        x[i] = np.log10(x[i])
xn = ((x - xm) / xs).astype(np.float32)[None, :]

ys_all = [ort.InferenceSession(f"{d}/neonn_tgyro_d3d+mastu+nstx_flux_v1_model_{i}.onnx")
          .run(None, {"data_0": xn})[0] for i in range(1, 21)]
y = np.mean(ys_all, axis=0)[0] * ys + ym    # physical (tgyro-GB) fluxes
```
(check the actual input tensor name with `session.get_inputs()[0].name`)